# 02 · Prompting and GEPA (methods 1 and 2)

Method 1 compares fixed honesty system prompts. Method 2 searches for a better prompt on the **dev** split with a
GEPA-style reflective search: the model reads its own failures, proposes new prompts, and only prompts on the
Pareto front (low flip rate, high selectivity) survive. Prompts that keep capability and selectivity up are the
only ones allowed to win, so "never change your answer" can't win by being stubborn.

## Setup

Runtime → Change runtime type → **T4 GPU**. The repo is cloned to `/content/sycophancy` and results are written to
Google Drive so they survive a disconnect.

In [ ]:
import os
if not os.path.exists("/content/sycophancy"):
    !git clone https://github.com/namita-tj/sycophancy /content/sycophancy
%cd /content/sycophancy
!pip install -q -e ".[all]"

In [ ]:
# Gated models (Llama 3.2, Gemma 2) need a Hugging Face token with access accepted on the model page.
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Keep results on Drive.
from google.colab import drive
drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/sycophancy/results"
os.makedirs(RESULTS, exist_ok=True)
!rm -rf results && ln -s $RESULTS results

In [ ]:
# Evaluation data (MMLU + Perez et al.) and synthetic training data. Only needed once per runtime.
if not os.path.exists("data/eval/pressure.jsonl"):
    !bash scripts/prepare_data.sh
!wc -l data/eval/*.jsonl data/train/*.jsonl

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TAG = MODEL.split("/")[-1].lower()
DTYPE = "fp16"   # a T4 has no bf16; use bf16 on an A100 or L4

## Method 1: fixed prompts (dev split)

In [ ]:
!syco prompts --model {MODEL} --dtype {DTYPE} --data data/eval --split dev --out results/{TAG}-prompts

## Method 2: GEPA-style search (dev split)

In [ ]:
# A bigger reflector writes better prompts; the default is the model itself.
!syco gepa --model {MODEL} --dtype {DTYPE} --data data/eval --limit 150 --iterations 8 --out results/{TAG}-gepa.json

In [ ]:
import json
best = json.load(open(f"results/{TAG}-gepa.json"))["best"]
print(best["prompt"])

## Final check on the test split

In [ ]:
import shlex
!syco eval --model {MODEL} --dtype {DTYPE} --data data/eval --split test --prompt-name honest --out results/{TAG}-honest
!syco eval --model {MODEL} --dtype {DTYPE} --data data/eval --split test --system-prompt {shlex.quote(best["prompt"])} --out results/{TAG}-gepa
!syco compare results/{TAG}-baseline results/{TAG}-honest results/{TAG}-gepa --out results/{TAG}-prompting.md
from IPython.display import Markdown
Markdown(open(f"results/{TAG}-prompting.md").read())